In [10]:
import pandas as pd
capacity_ts=pd.read_csv("../raw/mnre_re_cumulative_capacity_timeseries_2018_2025.csv")
dim_state=pd.read_csv("../clean/dim_state.csv")

In [3]:
print(capacity_ts.shape)
print(capacity_ts.columns.tolist())
print(capacity_ts.head(10).to_string(index=False))
print(capacity_ts.dtypes)

(42, 10)
['Region', 'State_UT', 'RE_MW_2017_18', 'RE_MW_2018_19', 'RE_MW_2019_20', 'RE_MW_2020_21', 'RE_MW_2021_22', 'RE_MW_2022_23', 'RE_MW_2023_24', 'RE_MW_2024_25']
         Region         State_UT  RE_MW_2017_18  RE_MW_2018_19  RE_MW_2019_20  RE_MW_2020_21  RE_MW_2021_22  RE_MW_2022_23  RE_MW_2023_24  RE_MW_2024_25
Northern Region          Haryana         510.49         519.11         547.19         762.51        1242.13        1362.09        1832.92        2449.94
Northern Region Himachal Pradesh       10514.30       10707.75       10768.91       10916.61       11303.49       11330.42       11356.16       12196.18
Northern Region  Jammu & Kashmir        3650.22        3664.10        3581.11        3588.11        3551.61        3556.12        3595.37        3624.42
Northern Region           Ladakh           0.00           0.00          89.00          89.00         136.44         137.79         139.79         142.59
Northern Region           Punjab        2626.98        2522.38     

In [4]:
print(capacity_ts["Region"].unique())
print("\nNumber of regions:", capacity_ts["Region"].nunique())

print("\nState/UT values:")
print(capacity_ts["State_UT"].to_list())

<ArrowStringArray>
['Northern Region', 'Eastern Region', 'Western Region', 'Southern Region']
Length: 4, dtype: str

Number of regions: 4

State/UT values:
['Haryana', 'Himachal Pradesh', 'Jammu & Kashmir', 'Ladakh', 'Punjab', 'Rajasthan', 'Uttar Pradesh', 'Uttarakhand', 'Chandigarh', 'Delhi', 'Total', 'Arunachal Pradesh', 'Assam', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Tripura', 'Total', 'Chhattisgarh', 'Goa', 'Gujarat', 'Madhya Pradesh', 'Maharashtra', 'Dadar & Nagar', 'Total', 'Bihar', 'Jharkhand', 'Odisha', 'Sikkim', 'West Bengal', 'Total', 'Andhra Pradesh', 'Karnataka', 'Kerala', 'Tamil Nadu', 'Telangana', 'Lakshadweep', 'Pondicherry', 'Total', 'Others', 'Grand Total']


In [11]:
ts_names = set(capacity_ts["State_UT"])
canonical_names = set(dim_state["state_name"])

print("In time series but not in dim_state:")
print(sorted(ts_names - canonical_names))

print("\nIn dim_state but not in time series:")
print(sorted(canonical_names - ts_names))

In time series but not in dim_state:
['Dadar & Nagar', 'Grand Total', 'Jammu & Kashmir', 'Others', 'Pondicherry', 'Total']

In dim_state but not in time series:
['Andaman and Nicobar Islands', 'Dadra and Nagar Haveli and Daman and Diu', 'Jammu and Kashmir', 'Puducherry']


In [12]:
print(
    capacity_ts.groupby("State_UT").size()
    .sort_values(ascending=False)
)

State_UT
Total                5
Andhra Pradesh       1
Assam                1
Bihar                1
Chandigarh           1
Arunachal Pradesh    1
Chhattisgarh         1
Dadar & Nagar        1
Goa                  1
Delhi                1
Gujarat              1
Haryana              1
Himachal Pradesh     1
Grand Total          1
Jharkhand            1
Karnataka            1
Kerala               1
Ladakh               1
Lakshadweep          1
Madhya Pradesh       1
Maharashtra          1
Jammu & Kashmir      1
Manipur              1
Meghalaya            1
Nagaland             1
Mizoram              1
Others               1
Pondicherry          1
Punjab               1
Odisha               1
Rajasthan            1
Sikkim               1
Tamil Nadu           1
Telangana            1
Tripura              1
Uttar Pradesh        1
Uttarakhand          1
West Bengal          1
dtype: int64


In [13]:
print(
    capacity_ts[
        capacity_ts["State_UT"].isin([
            "Dadar & Nagar",
            "Jammu & Kashmir",
            "Pondicherry",
            "Others",
            "Total",
            "Grand Total"
        ])
    ].to_string(index=False)
)

         Region        State_UT  RE_MW_2017_18  RE_MW_2018_19  RE_MW_2019_20  RE_MW_2020_21  RE_MW_2021_22  RE_MW_2022_23  RE_MW_2023_24  RE_MW_2024_25
Northern Region Jammu & Kashmir        3650.22        3664.10        3581.11        3588.11        3551.61        3556.12        3595.37        3624.42
Northern Region           Total       32567.47       34019.28       36286.95       38357.09       46646.97       52710.14       58267.19       67530.82
 Eastern Region           Total        1662.72        1795.35        2122.39        2429.56        2500.55        2566.81        2601.84        2656.41
 Western Region   Dadar & Nagar          16.07          19.93          25.32          46.01          46.18          46.47          46.47          51.87
 Western Region           Total       28106.36       30859.07       33687.29       37054.55       41041.46       46863.60       56100.54       68560.95
 Eastern Region           Total        7013.75        7391.28        7454.03        7561

In [14]:
capacity_ts_clean = capacity_ts[
    ~capacity_ts["State_UT"].isin([
        "Total",
        "Grand Total",
        "Others"
    ])
].copy()
capacity_ts_clean["state_name"] = capacity_ts_clean["State_UT"].replace({
    "Jammu & Kashmir": "Jammu and Kashmir",
    "Pondicherry": "Puducherry",
    "Dadar & Nagar": "Dadra and Nagar Haveli and Daman and Diu"
})

In [15]:
ts_names = set(capacity_ts_clean["state_name"])
canonical_names = set(dim_state["state_name"])

print("Missing from time series:")
print(sorted(canonical_names - ts_names))

print("\nExtra in time series:")
print(sorted(ts_names - canonical_names))

print("\nRows:", len(capacity_ts_clean))

Missing from time series:
['Andaman and Nicobar Islands']

Extra in time series:
[]

Rows: 35


In [16]:
capacity_ts_clean = capacity_ts_clean.merge(
    dim_state[["state_id", "state_name"]],
    on="state_name",
    how="left",
    validate="one_to_one"
)

In [17]:
print(capacity_ts_clean.shape)

print("Missing state IDs:")
print(capacity_ts_clean["state_id"].isna().sum())

print("Unique state IDs:")
print(capacity_ts_clean["state_id"].nunique())

(35, 12)
Missing state IDs:
0
Unique state IDs:
35


In [18]:
year_columns = [
    "RE_MW_2017_18",
    "RE_MW_2018_19",
    "RE_MW_2019_20",
    "RE_MW_2020_21",
    "RE_MW_2021_22",
    "RE_MW_2022_23",
    "RE_MW_2023_24",
    "RE_MW_2024_25"
]

capacity_ts_long = capacity_ts_clean.melt(
    id_vars=[
        "state_id",
        "state_name",
        "Region"
    ],
    value_vars=year_columns,
    var_name="fiscal_year",
    value_name="cumulative_re_capacity_mw"
)

In [19]:
capacity_ts_long["fiscal_year"] = (
    capacity_ts_long["fiscal_year"]
    .str.replace("RE_MW_", "", regex=False)
)

In [21]:
print(capacity_ts_long.shape)
print(capacity_ts_long.columns.tolist())
print(capacity_ts_long.head(200).to_string(index=False))

(280, 5)
['state_id', 'state_name', 'Region', 'fiscal_year', 'cumulative_re_capacity_mw']
state_id                               state_name          Region fiscal_year  cumulative_re_capacity_mw
   IN-HR                                  Haryana Northern Region     2017_18                     510.49
   IN-HP                         Himachal Pradesh Northern Region     2017_18                   10514.30
   IN-JK                        Jammu and Kashmir Northern Region     2017_18                    3650.22
   IN-LA                                   Ladakh Northern Region     2017_18                       0.00
   IN-PB                                   Punjab Northern Region     2017_18                    2626.98
   IN-RJ                                Rajasthan Northern Region     2017_18                    7289.36
   IN-UP                            Uttar Pradesh Northern Region     2017_18                    3447.25
   IN-UT                              Uttarakhand Northern Region     

In [22]:
capacity_ts_long = capacity_ts_long.rename(
    columns={"Region": "region"}
)

In [23]:
capacity_ts_long = capacity_ts_long[
    [
        "state_id",
        "state_name",
        "region",
        "fiscal_year",
        "cumulative_re_capacity_mw"
    ]
]

In [25]:
capacity_ts_long.to_csv(
    "../clean/fact_re_capacity_long.csv",
    index=False
)